# **Kaggle – DataTops®**
Tu TA ha decidido cambiar de aires y, por eso, ha comprado una tienda de portátiles. Sin embargo, su única especialidad es Data Science, por lo que ha decidido crear un modelo de ML para establecer los mejores precios.

¿Podrías ayudar a tu profe a mejorar ese modelo?

## Aspectos importantes
- Última submission:
    - Mañana: 17 de febrero a las 5pm
    - Tarde: 19 de febrero a las 5pm
- **Enlace de la competición**: https://www.kaggle.com/t/c5cc87b50c4b4770bdc8f5acbe15577d
- **Requisito**: Estar registrado en [Kaggle](https://www.kaggle.com/)

## Métrica:
El error cuadrático medio (RMSE, por sus siglas en inglés) es una medida de la desviación estándar de los residuos (errores de predicción). Los residuos representan la diferencia entre los valores observados y los valores predichos por el modelo. El RMSE indica qué tan dispersos están estos errores: cuanto menor es el RMSE, más cercanas están las predicciones a los valores reales. En otras palabras, el RMSE mide qué tan bien se ajusta la línea de regresión a los datos.


$$ RMSE = \sqrt{\frac{1}{n}\Sigma_{i=1}^{n}{\Big(\frac{d_i -f_i}{\sigma_i}\Big)^2}}$$


## 1. Librerías

In [1261]:
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
import urllib.request

import re

## 2. Datos

In [1262]:
# Para que funcione necesitas bajarte los archivos de datos de Kaggle
df = pd.read_csv("./data/train.csv", index_col=0)

### 2.1 Exploración de los datos

In [1263]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 912 entries, 755 to 229
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Company           912 non-null    object 
 1   Product           912 non-null    object 
 2   TypeName          912 non-null    object 
 3   Inches            912 non-null    float64
 4   ScreenResolution  912 non-null    object 
 5   Cpu               912 non-null    object 
 6   Ram               912 non-null    object 
 7   Memory            912 non-null    object 
 8   Gpu               912 non-null    object 
 9   OpSys             912 non-null    object 
 10  Weight            912 non-null    object 
 11  Price_in_euros    912 non-null    float64
dtypes: float64(2), object(10)
memory usage: 92.6+ KB


In [1264]:
df.head()

,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_in_euros
laptop_ID,,,,,,,,,,,,
755,HP,250 G6,Notebook,15.6,Full HD 1920x1080,Intel Core i3 6006U 2GHz,8GB,256GB SSD,Intel HD Graphics 520,Windows 10,1.86kg,539.00
618,Dell,Inspiron 7559,Gaming,15.6,Full HD 1920x1080,Intel Core i7 6700HQ 2.6GHz,16GB,1TB HDD,Nvidia GeForce GTX 960<U+039C>,Windows 10,2.59kg,879.01
909,HP,ProBook 450,Notebook,15.6,Full HD 1920x1080,Intel Core i7 7500U 2.7GHz,8GB,1TB HDD,Nvidia GeForce 930MX,Windows 10,2.04kg,900.00
2,Apple,Macbook Air,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,898.94
286,Dell,Inspiron 3567,Notebook,15.6,Full HD 1920x1080,Intel Core i3 6006U 2.0GHz,4GB,1TB HDD,AMD Radeon R5 M430,Linux,2.25kg,428.00


In [1265]:
df.tail()

,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_in_euros
laptop_ID,,,,,,,,,,,,
28,Dell,Inspiron 5570,Notebook,15.6,Full HD 1920x1080,Intel Core i5 8250U 1.6GHz,8GB,256GB SSD,AMD Radeon 530,Windows 10,2.2kg,800.00
1160,HP,Spectre Pro,2 in 1 Convertible,13.3,Full HD / Touchscreen 1920x1080,Intel Core i5 6300U 2.4GHz,8GB,256GB SSD,Intel HD Graphics 520,Windows 10,1.48kg,1629.00
78,Lenovo,IdeaPad 320-15IKBN,Notebook,15.6,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,2TB HDD,Intel HD Graphics 620,No OS,2.2kg,519.00
23,HP,255 G6,Notebook,15.6,1366x768,AMD E-Series E2-9000e 1.5GHz,4GB,500GB HDD,AMD Radeon R2,No OS,1.86kg,258.00
229,Dell,Alienware 17,Gaming,17.3,IPS Panel Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,16GB,256GB SSD + 1TB HDD,Nvidia GeForce GTX 1060,Windows 10,4.42kg,2456.34


In [1266]:
df.describe()

,Inches,Price_in_euros
count,912.000000,912.000000
mean,14.981579,1111.724090
std,1.436719,687.959172
min,10.100000,174.000000
25%,14.000000,589.000000
50%,15.600000,978.000000
75%,15.600000,1483.942500
max,18.400000,6099.000000


In [1267]:
for col in df.columns:
    print(col, '->', df[col].nunique())

Company -> 19
Product -> 480
TypeName -> 6
Inches -> 17
ScreenResolution -> 36
Cpu -> 107
Ram -> 9
Memory -> 37
Gpu -> 93
OpSys -> 9
Weight -> 165
Price_in_euros -> 603


In [1268]:
df['Company'].value_counts()

Company
Lenovo       202
Dell         197
HP           194
Asus         121
Acer          74
MSI           37
Toshiba       34
Apple         17
Razer          6
Mediacom       6
Samsung        5
Microsoft      5
Xiaomi         3
Huawei         2
Chuwi          2
Google         2
Vero           2
Fujitsu        2
LG             1
Name: count, dtype: int64

In [1269]:
otras_company = df['Company'].unique()[df['Company'].value_counts(sort=False) < 5]

df['company_t'] = df['Company']
df.loc[df['company_t'].isin(otras_company), 'company_t'] = 'Otros'
df['company_t'].value_counts()

company_t
Lenovo       202
Dell         197
HP           194
Asus         121
Acer          74
MSI           37
Toshiba       34
Apple         17
Otros         14
Razer          6
Mediacom       6
Samsung        5
Microsoft      5
Name: count, dtype: int64

In [1270]:
df['TypeName'].value_counts()

TypeName
Notebook              509
Gaming                143
Ultrabook             141
2 in 1 Convertible     80
Workstation            20
Netbook                19
Name: count, dtype: int64

In [1271]:
df['Product'].value_counts()

Product
XPS 13                                   23
Inspiron 3567                            22
Legion Y520-15IKBN                       15
Vostro 3568                              14
ProBook 450                              13
                                         ..
Inspiron 7773                             1
ENVY -                                    1
Latitude E7270                            1
Rog GL552VW-CN470T                        1
15-BS026nv (i5-7200U/8GB/256GB/Radeon     1
Name: count, Length: 480, dtype: int64

In [1272]:
df['product_len'] = df['Product'].str.len()
df['product_words'] = df['Product'].str.split(' ').str.len()

In [1273]:
df['Inches'].value_counts()

Inches
15.6    453
14.0    150
13.3    114
17.3    113
11.6     28
12.5     27
13.5      5
12.0      4
15.0      3
15.4      3
13.9      3
10.1      2
13.0      2
12.3      2
14.1      1
11.3      1
18.4      1
Name: count, dtype: int64

In [1274]:
df['ScreenResolution'].value_counts()

ScreenResolution
Full HD 1920x1080                                349
1366x768                                         211
IPS Panel Full HD 1920x1080                      163
IPS Panel Full HD / Touchscreen 1920x1080         32
Full HD / Touchscreen 1920x1080                   30
1600x900                                          14
Quad HD+ / Touchscreen 3200x1800                  11
Touchscreen 1366x768                              11
IPS Panel 4K Ultra HD / Touchscreen 3840x2160     10
4K Ultra HD / Touchscreen 3840x2160                7
Touchscreen 2560x1440                              6
IPS Panel Quad HD+ / Touchscreen 3200x1800         6
IPS Panel 4K Ultra HD 3840x2160                    5
IPS Panel Retina Display 2560x1600                 5
Touchscreen 2256x1504                              5
1440x900                                           4
IPS Panel Touchscreen 2560x1440                    4
IPS Panel Retina Display 2304x1440                 4
IPS Panel 1366x768           

In [1275]:
df['r_ancho'] = df['ScreenResolution'].str.extract(r'(\d+)x\d+').astype(int)
df['r_alto'] = df['ScreenResolution'].str.extract(r'\d+x(\d+)').astype(int)
df['r_prop'] = df['r_ancho'] / df['r_alto']

df['is_ips_panel'] = df['ScreenResolution'].str.contains(r'ips\s*panel', case=False)
df['is_touchscreen'] = df['ScreenResolution'].str.contains(r'touchscreen', case=False)
# df['is_retina_display'] = df['ScreenResolution'].str.contains(r'retina\s*display', case=False) # Solo para Apple, descarto

In [1276]:
df['Cpu'].value_counts()

Cpu
Intel Core i5 7200U 2.5GHz              124
Intel Core i7 7700HQ 2.8GHz             105
Intel Core i7 7500U 2.7GHz               97
Intel Core i5 8250U 1.6GHz               52
Intel Core i7 8550U 1.8GHz               47
                                       ... 
Intel Core i3 6006U 2.2GHz                1
Intel Atom Z8350 1.92GHz                  1
Intel Core i5 7200U 2.50GHz               1
AMD A6-Series 7310 2GHz                   1
Intel Pentium Dual Core N4200 1.1GHz      1
Name: count, Length: 107, dtype: int64

In [1277]:
df['ghz'] = df['Cpu'].transform(lambda x: x.split(' ')[-1].replace('GHz', ''))

df['is_intel'] = df['Cpu'].str.contains(r'intel', case=False) # Separo entre Intel y AMD

In [1278]:
def cpu_categoria(texto):
    texto = texto.lower()
    if 'core i3' in texto:
        return 3
    if 'core i5' in texto:
        return 4
    if 'core i7' in texto:
        return 4
    if 'core i9' in texto:
        return 5
    if 'pentium dual' in texto:
        return 2
    if 'pentium quad' in texto:
        return 2
    if 'celeron dual' in texto:
        return 2
    if 'celeron quad' in texto:
        return 2
    if 'core m' in texto:
        return 1
    if 'atom' in texto:
        return 1
    if 'xeon' in texto:
        return 5
    if 'ryzen' in texto:
        return 4
    if 'amd a' in texto:
        return 3
    if 'amd e' in texto:
        return 3
    else:
        return 3
    
    
df['cpu_cat'] = df['Cpu'].transform(cpu_categoria)
df['cpu_cat'].value_counts(dropna=False)

cpu_cat
4    659
3    132
2     94
1     25
5      2
Name: count, dtype: int64

In [1279]:
def cpu_modelo(texto):
    modelo = re.search(r'([a-zA-Z]*\d{3,5}[a-zA-Z]*)', texto)
    if not modelo:
        modelo = re.search(r'(\d+Y\d+)', texto)
        
    if modelo:
        return modelo.group(0)

df['cpu_model'] = df['Cpu'].transform(cpu_modelo)
df.loc[df['cpu_cat'] == 5, 'cpu_model'].unique()

array(['1505M', '1535M'], dtype=object)

In [1280]:
def cpu_year_intel(texto):
    texto = texto if texto else ''
    digito = re.search(r'^\D*(\d)', texto)
    if digito:
        digito = int(digito.group(0))
        if digito > 5:
            return digito + 9
        
def cpu_year_amd(texto):
    texto = texto if texto else ''
    digito = re.search(r'^\D*(\d)', texto)
    if digito:
        digito = int(digito.group(0))
        if digito > 5:
            return digito + 8

In [1281]:
df.loc[(df['is_intel']) & ((df['cpu_cat'] == 3) | (df['cpu_cat'] == 4)), 'cpu_year'] = df.loc[(df['is_intel']) & ((df['cpu_cat'] == 3) | (df['cpu_cat'] == 4)), 'cpu_model'].transform(cpu_year_intel)
df.loc[~df['is_intel'], 'cpu_year'] = df.loc[~df['is_intel'], 'cpu_model'].transform(cpu_year_amd)
df.loc[(~df['cpu_model'].isna()) & (df['cpu_year'].isna()), 'cpu_year'] = 15
df['cpu_year'].value_counts(dropna=False)

cpu_year
16.0    418
15.0    341
17.0    134
NaN      17
14.0      2
Name: count, dtype: int64

In [1282]:
def cpu_letra(texto):
    texto = texto if texto else ''
    letra = re.findall(r'\D+', texto)
    if letra:
        return letra[0]
    else:
        return np.nan

In [1283]:
df['cpu_letra'] = df['cpu_model'].apply(cpu_letra)

# # df['cpu_letra'].unique()
# df['cpu_letra'].value_counts(dropna=False, sort=False)

otras_cpu_letra = df['cpu_letra'].unique()[df['cpu_letra'].value_counts(dropna=False, sort=False) < 5]

df.loc[df['cpu_letra'].isin(otras_cpu_letra), 'cpu_letra'] = np.nan
df['cpu_letra'].value_counts(dropna=False)

cpu_letra
U      550
HQ     177
N       86
NaN     51
Y       19
Z       10
P       10
HK       9
Name: count, dtype: int64

In [1284]:
df['Ram'].value_counts()

Ram
8GB     434
4GB     267
16GB    136
6GB      24
2GB      20
12GB     19
32GB     10
64GB      1
24GB      1
Name: count, dtype: int64

In [1285]:
df['ram_gb'] = df['Ram'].str.extract(r'(\d+)')

df['ram_gb'].value_counts()

ram_gb
8     434
4     267
16    136
6      24
2      20
12     19
32     10
64      1
24      1
Name: count, dtype: int64

In [1286]:
df['Memory'].value_counts()

Memory
256GB SSD                        282
1TB HDD                          152
500GB HDD                         92
512GB SSD                         83
128GB SSD +  1TB HDD              67
128GB SSD                         54
256GB SSD +  1TB HDD              52
32GB Flash Storage                33
1TB SSD                           12
64GB Flash Storage                11
2TB HDD                            8
512GB SSD +  1TB HDD               8
256GB Flash Storage                7
256GB SSD +  2TB HDD               6
16GB Flash Storage                 6
1.0TB Hybrid                       5
32GB SSD                           5
128GB Flash Storage                4
180GB SSD                          3
16GB SSD                           3
1TB SSD +  1TB HDD                 2
512GB SSD +  2TB HDD               2
256GB SSD +  256GB SSD             1
128GB SSD +  2TB HDD               1
512GB SSD +  512GB SSD             1
64GB Flash Storage +  1TB HDD      1
64GB SSD                       

In [1287]:
def memory(lista):
    tipo = {'SSD': 0, 'HDD': 0, 'Flash': 0, 'Hybrid': 0}
    for elemento in lista:
        numero = float(re.match(r'.*(\d+)', elemento).group(0))
        if 'TB' in elemento:
            numero *= 1024
        for key in tipo.keys():
            if key in elemento:
                tipo[key] += numero
    
    return pd.Series(tipo)

In [1288]:
df['drives'] = df['Memory'].str.split('+')

df['n_drives'] = df['drives'].str.len()

In [1289]:
df[['ssd', 'hdd', 'flash', 'hybrid']] = df['drives'].apply(memory)
df['mem'] = df['ssd'] + df['hdd'] + df['flash'] + df['hybrid']

In [1290]:
df['Gpu'].value_counts()

Gpu
Intel HD Graphics 620       185
Intel HD Graphics 520       125
Intel UHD Graphics 620       52
Nvidia GeForce GTX 1050      48
Nvidia GeForce 940MX         31
                           ... 
AMD Radeon RX 540             1
Nvidia Quadro M2000M          1
Nvidia GeForce GTX 940M       1
AMD Radeon R5 520             1
Nvidia GeForce GTX 1070M      1
Name: count, Length: 93, dtype: int64

In [ ]:
def nvidia_categoria(texto):
    texto = texto.lower()
    if 'quadro' in texto:
        return 4
    if '70' or '80' in texto:
        return 3
    if '50' or '60' in texto:
        return 2
    else:
        return 1
    
def amd_categoria(texto):
    texto = texto.lower()
    if 'firepro' in texto:
        return 4
    if '70' or '80' in texto:
        return 3
    if '50' or '60' in texto:
        return 2
    else:
        return 1

In [1303]:
df['gpu_nvidia'] = df['Gpu'].str.contains(r'nvidia', case=False)
df['gpu_amd'] = df['Gpu'].str.contains(r'amd', case=False)

# df['gpu_amd'] = df['Gpu'].str.contains(
df['gpu_amd'] = df['Gpu'].str.contains(r'r\d$|r\d\sg', case=False)

In [1252]:
df.loc[df['gpu_nvidia'], 'Gpu'].unique()

array(['Nvidia GeForce GTX 960<U+039C>', 'Nvidia GeForce 930MX',
       'Nvidia GeForce 940MX', 'Nvidia GeForce GTX 1050 Ti',
       'Nvidia GeForce GTX 1060', 'Nvidia GeForce GTX 960M',
       'Nvidia GeForce GTX 1050', 'Nvidia Quadro M1200',
       'Nvidia GeForce GTX 1080', 'Nvidia GeForce GTX 1070',
       'Nvidia Quadro M620', 'Nvidia Quadro M500M',
       'Nvidia GeForce 920MX ', 'Nvidia GeForce GTX 980M',
       'Nvidia GeForce GT 940MX', 'Nvidia GeForce GTX 970M',
       'Nvidia GeForce MX150', 'Nvidia GeForce GTX 1050Ti',
       'Nvidia GeForce MX130', 'Nvidia GeForce 930M',
       'Nvidia GeForce 920MX', 'Nvidia GeForce GTX 1050M',
       'Nvidia GeForce GTX 950M', 'Nvidia Quadro M1000M',
       'Nvidia GeForce 150MX', 'Nvidia GeForce GTX 940MX',
       'Nvidia GeForce GTX 980 ', 'Nvidia GeForce GTX 960',
       'Nvidia GeForce 920M', 'Nvidia GeForce GTX 965M',
       'Nvidia GTX 980 SLI', 'Nvidia Quadro M3000M',
       'Nvidia Quadro M2200M', 'Nvidia GeForce GTX1050 Ti',
   

In [1253]:
df.loc[df['gpu_amd'], 'Gpu'].unique()

array(['AMD Radeon R5 M430', 'AMD Radeon R7 M445', 'AMD Radeon R7 M460',
       'AMD FirePro W6150M', 'AMD Radeon RX 580', 'AMD Radeon R5 M330',
       'AMD Radeon 530', 'AMD Radeon 520', 'AMD Radeon R5 M420',
       'AMD Radeon RX 560', 'AMD FirePro W4190M', 'AMD Radeon RX 550',
       'AMD Radeon 540', 'AMD Radeon R4 Graphics', 'AMD Radeon R7',
       'AMD Radeon R2 Graphics', 'AMD Radeon R7 M440', 'AMD Radeon R2',
       'AMD Radeon R5', 'AMD FirePro W5130M', 'AMD Radeon R5 M420X',
       'AMD Radeon R5 M315', 'AMD Radeon R7 M365X', 'AMD Radeon R5 430',
       'AMD FirePro W4190M ', 'AMD Radeon Pro 455', 'AMD R17M-M1-70',
       'AMD Radeon R4', 'AMD R4 Graphics', 'AMD Radeon RX 540',
       'AMD Radeon Pro 555', 'AMD Radeon R5 520'], dtype=object)

In [1251]:
df.loc[~(df['gpu_amd'] | df['gpu_nvidia']), 'Gpu'].value_counts()

Gpu
Intel HD Graphics 620           185
Intel HD Graphics 520           125
Intel UHD Graphics 620           52
Intel HD Graphics 400            30
Intel HD Graphics 500            27
Intel HD Graphics                25
Intel HD Graphics 515            12
Intel HD Graphics 615            10
Intel HD Graphics 505             8
Intel HD Graphics 405             8
Intel Iris Plus Graphics 640      6
Intel HD Graphics 6000            5
Intel HD Graphics 510             4
Intel Iris Plus Graphics 650      2
Intel HD Graphics 630             2
Intel Iris Graphics 540           2
Intel Iris Graphics 550           1
Intel Graphics 620                1
Intel HD Graphics 5300            1
Intel Iris Pro Graphics           1
Intel HD Graphics 620             1
Intel HD Graphics 540             1
Name: count, dtype: int64

In [1240]:
df['Gpu'].unique()

array(['Intel HD Graphics 520', 'Nvidia GeForce GTX 960<U+039C>',
       'Nvidia GeForce 930MX', 'Intel HD Graphics 6000',
       'AMD Radeon R5 M430', 'Intel HD Graphics 620',
       'Nvidia GeForce 940MX', 'Nvidia GeForce GTX 1050 Ti',
       'Intel Iris Graphics 550', 'Intel HD Graphics 505',
       'Intel UHD Graphics 620', 'Intel HD Graphics 405',
       'Nvidia GeForce GTX 1060', 'Nvidia GeForce GTX 960M',
       'Intel HD Graphics 400', 'AMD Radeon R7 M445',
       'AMD Radeon R7 M460', 'Intel HD Graphics', 'AMD FirePro W6150M',
       'Nvidia GeForce GTX 1050', 'Nvidia Quadro M1200',
       'AMD Radeon RX 580', 'Nvidia GeForce GTX 1080',
       'AMD Radeon R5 M330', 'Nvidia GeForce GTX 1070',
       'Intel HD Graphics 615', 'AMD Radeon 530', 'AMD Radeon 520',
       'Nvidia Quadro M620', 'Intel Iris Plus Graphics 640',
       'Nvidia Quadro M500M', 'Intel HD Graphics 510',
       'Intel Iris Plus Graphics 650', 'Nvidia GeForce 920MX ',
       'Intel HD Graphics 500', 'AMD Radeo

In [36]:
df['OpSys'].value_counts()

OpSys
Windows 10      746
Linux            47
No OS            44
Windows 7        32
Chrome OS        20
macOS            11
Mac OS X          6
Windows 10 S      4
Android           2
Name: count, dtype: int64

In [37]:
df['Weight'].value_counts()

Weight
2.2kg     86
2.1kg     40
2.3kg     35
2.4kg     31
2kg       30
          ..
1.41kg     1
4kg        1
2.72kg     1
1.94kg     1
1.79kg     1
Name: count, Length: 158, dtype: int64

### 2.3 Definir X e y

In [10]:
X = df.drop(['Price_euros'], axis=1)
y = df['Price_euros'].copy()
X.shape

(912, 13)

In [11]:
y.shape

(912,)

### 2.4 Dividir X_train, X_test, y_train, y_test

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state = 42)

In [13]:
X_train

,id,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight
25,829,41,Asus,X540UA-DM186 (i3-6006U/4GB/1TB/FHD/Linux),Notebook,15.6,Full HD 1920x1080,Intel Core i3 6006U 2GHz,4GB,1TB HDD,Intel HD Graphics 620,Linux,2kg
84,788,127,Acer,Aspire 3,Notebook,15.6,1366x768,AMD A9-Series 9420 3GHz,4GB,256GB SSD,AMD Radeon R5,Windows 10,2.1kg
10,851,1243,Asus,X540SA-RBPDN09 (N3710/4GB/1TB/W10),Notebook,15.6,1366x768,Intel Pentium Quad Core N3710 1.6GHz,4GB,1TB HDD,Intel HD Graphics 405,Windows 10,2.65kg
342,126,105,Dell,Inspiron 3576,Notebook,15.6,Full HD 1920x1080,Intel Core i5 8250U 1.6GHz,8GB,1TB HDD,AMD Radeon 520,Linux,2.2kg
890,223,578,HP,14-am079na (N3710/8GB/2TB/W10),Notebook,14.0,1366x768,Intel Pentium Quad Core N3710 1.6GHz,8GB,2TB HDD,Intel HD Graphics 405,Windows 10,1.94kg
...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,555,267,HP,ProBook 450,Notebook,15.6,IPS Panel Full HD 1920x1080,Intel Core i5 8250U 1.6GHz,4GB,500GB HDD,Intel HD Graphics 620,Windows 10,2.1kg
270,308,610,MSI,Laptop MSI,Gaming,17.3,Full HD 1920x1080,Intel Core i7 6820HK 2.7GHz,16GB,128GB SSD + 1TB HDD,Nvidia GeForce GTX 970M,Windows 10,4.14kg
860,281,1026,HP,Elitebook 840,Notebook,14.0,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,4GB,256GB SSD,Intel HD Graphics 620,Windows 10,1.48kg
435,729,363,Dell,Inspiron 7577,Gaming,15.6,Full HD 1920x1080,Intel Core i5 7300HQ 2.5GHz,8GB,1TB HDD,Nvidia GeForce GTX 1050,Windows 10,2.65kg


In [14]:
y_train

25      389.0
84      451.0
10      309.0
342     647.0
890     389.0
        ...  
106     722.0
270    2199.0
860    1590.0
435     999.0
102    1799.0
Name: Price_euros, Length: 729, dtype: float64

## 3. Procesado de datos

Nuestro target es la columna `Price_in_euros`

-----------------------------------------------------------------------------------------------------------------

## 4. Modelado

### 4.1 Baseline de modelos


### 4.2 Sacar métricas, valorar los modelos

Recuerda que en la competición se va a evaluar con la métrica de ``RMSE``.

### 4.3 Optimización (up to you 🫰🏻)

-----------------------------------------------------------------

## Una vez listo el modelo, toca predecir ``test.csv``

**RECUERDA: APLICAR LAS TRANSFORMACIONES QUE HAYAS REALIZADO EN `train.csv` a `test.csv`.**


Véase:
- Estandarización/Normalización
- Eliminación de Outliers
- Eliminación de columnas
- Creación de columnas nuevas
- Gestión de valores nulos
- Y un largo etcétera de técnicas que como Data Scientist hayas considerado las mejores para tu dataset.

## 1. Carga los datos de `test.csv` para predecir.


In [15]:
X_pred = pd.read_csv("./data/test.csv")
X_pred.head()

,id,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight
0,181,1098,HP,Spectre x360,Ultrabook,13.3,IPS Panel 4K Ultra HD 3840x2160,Intel Core i7 7500U 2.7GHz,16GB,512GB SSD,Intel HD Graphics 620,Windows 10,1.3kg
1,708,330,Acer,Aspire 5,Notebook,15.6,1366x768,AMD A12-Series 9720P 2.7GHz,8GB,256GB SSD,AMD Radeon RX 540,Windows 10,2.2kg
2,862,1260,Acer,Aspire ES1-572,Notebook,15.6,1366x768,Intel Core i3 6006U 2.0GHz,4GB,500GB HDD,Intel HD Graphics 520,Linux,2.4kg
3,1064,1137,HP,EliteBook 1040,Notebook,14.0,Full HD 1920x1080,Intel Core i5 6200U 2.3GHz,8GB,256GB SSD,Intel HD Graphics 520,Windows 7,1.43kg
4,702,1015,HP,ENVY -,Notebook,13.3,IPS Panel Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,Windows 10,1.34kg


In [16]:
X_pred.tail()

,id,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight
386,1281,145,Lenovo,Legion Y520-15IKBN,Gaming,15.6,IPS Panel Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,8GB,256GB SSD,Nvidia GeForce GTX 1050M,No OS,2.4kg
387,524,1195,Lenovo,IdeaPad Y700-15ISK,Gaming,15.6,IPS Panel Full HD 1920x1080,Intel Core i7 6700HQ 2.6GHz,16GB,512GB SSD,Nvidia GeForce GTX 960,Windows 10,3.31kg
388,1015,1070,HP,250 G5,Notebook,15.6,1366x768,Intel Core i5 7200U 2.5GHz,4GB,500GB HDD,Intel HD Graphics 620,No OS,1.96kg
389,1236,104,HP,15-bw000nv (E2-9000e/4GB/500GB/Radeon,Notebook,15.6,Full HD 1920x1080,AMD E-Series E2-9000e 1.5GHz,4GB,500GB HDD,AMD Radeon R2,Windows 10,2.1kg
390,1036,258,Lenovo,Yoga 920-13IKB,2 in 1 Convertible,13.9,IPS Panel Full HD / Touchscreen 1920x1080,Intel Core i7 8550U 1.8GHz,8GB,512GB SSD,Intel UHD Graphics 620,Windows 10,1.37kg


In [17]:
X_pred.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 391 entries, 0 to 390
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                391 non-null    int64  
 1   laptop_ID         391 non-null    int64  
 2   Company           391 non-null    object 
 3   Product           391 non-null    object 
 4   TypeName          391 non-null    object 
 5   Inches            391 non-null    float64
 6   ScreenResolution  391 non-null    object 
 7   Cpu               391 non-null    object 
 8   Ram               391 non-null    object 
 9   Memory            391 non-null    object 
 10  Gpu               391 non-null    object 
 11  OpSys             391 non-null    object 
 12  Weight            391 non-null    object 
dtypes: float64(1), int64(2), object(10)
memory usage: 39.8+ KB


 ## 2. Replicar el procesado para ``test.csv``

In [18]:
X_pred

,id,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight
0,181,1098,HP,Spectre x360,Ultrabook,13.3,IPS Panel 4K Ultra HD 3840x2160,Intel Core i7 7500U 2.7GHz,16GB,512GB SSD,Intel HD Graphics 620,Windows 10,1.3kg
1,708,330,Acer,Aspire 5,Notebook,15.6,1366x768,AMD A12-Series 9720P 2.7GHz,8GB,256GB SSD,AMD Radeon RX 540,Windows 10,2.2kg
2,862,1260,Acer,Aspire ES1-572,Notebook,15.6,1366x768,Intel Core i3 6006U 2.0GHz,4GB,500GB HDD,Intel HD Graphics 520,Linux,2.4kg
3,1064,1137,HP,EliteBook 1040,Notebook,14.0,Full HD 1920x1080,Intel Core i5 6200U 2.3GHz,8GB,256GB SSD,Intel HD Graphics 520,Windows 7,1.43kg
4,702,1015,HP,ENVY -,Notebook,13.3,IPS Panel Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,Windows 10,1.34kg
...,...,...,...,...,...,...,...,...,...,...,...,...,...
386,1281,145,Lenovo,Legion Y520-15IKBN,Gaming,15.6,IPS Panel Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,8GB,256GB SSD,Nvidia GeForce GTX 1050M,No OS,2.4kg
387,524,1195,Lenovo,IdeaPad Y700-15ISK,Gaming,15.6,IPS Panel Full HD 1920x1080,Intel Core i7 6700HQ 2.6GHz,16GB,512GB SSD,Nvidia GeForce GTX 960,Windows 10,3.31kg
388,1015,1070,HP,250 G5,Notebook,15.6,1366x768,Intel Core i5 7200U 2.5GHz,4GB,500GB HDD,Intel HD Graphics 620,No OS,1.96kg
389,1236,104,HP,15-bw000nv (E2-9000e/4GB/500GB/Radeon,Notebook,15.6,Full HD 1920x1080,AMD E-Series E2-9000e 1.5GHz,4GB,500GB HDD,AMD Radeon R2,Windows 10,2.1kg


In [19]:
predictions_submit = model.predict(X_pred)
predictions_submit

NameError: name 'model' is not defined

**¡OJO! ¿Por qué me da error?**

IMPORTANTE:

- SI EL ARRAY CON EL QUE HICISTEIS `.fit()` ERA DE 4 COLUMNAS, PARA `.predict()` DEBEN SER LAS MISMAS
- SI AL ARRAY CON EL QUE HICISTEIS `.fit()` LO NORMALIZASTEIS, PARA `.predict()` DEBÉIS NORMALIZARLO
- TODO IGUAL SALVO **BORRAR FILAS**, EL NÚMERO DE ROWS SE DEBE MANTENER EN ESTE SET, PUES LA PREDICCIÓN DEBE TENER **391 FILAS**, SI O SI

**Entonces, si al cargar los datos de ``train.csv`` usaste `index_col=0`, ¿tendré que hacer lo también para el `test.csv`?**

In [ ]:
# ¿Qué opináis?
# ¿Sí, no?

![wow.jpeg](attachment:wow.jpeg)

## 3. **¿Qué es lo que subirás a Kaggle?**

**Para subir a Kaggle la predicción esta tendrá que tener una forma específica.**

En este caso, la **MISMA** forma que `sample_submission.csv`.

In [20]:
sample = pd.read_csv("data/sample_submission.csv")

In [21]:
sample.head()

,id,Price_euros
0,1014,752.0
1,845,499.0
2,1151,1747.0
3,1265,245.0
4,573,1179.0


In [22]:
sample.shape

(391, 2)

## 4. Mete tus predicciones en un dataframe llamado ``submission``.

In [ ]:
#¿Cómo creamos la submission?
submission = pd.DataFrame()

In [ ]:
submission.head()

In [ ]:
submission.shape

## 5. Pásale el CHEQUEADOR para comprobar que efectivamente está listo para subir a Kaggle.

In [ ]:
def chequeador(df_to_submit):
    """
    Esta función se asegura de que tu submission tenga la forma requerida por Kaggle.

    Si es así, se guardará el dataframe en un `csv` y estará listo para subir a Kaggle.

    Si no, LEE EL MENSAJE Y HAZLE CASO.

    Si aún no:
    - apaga tu ordenador,
    - date una vuelta,
    - enciendelo otra vez,
    - abre este notebook y
    - leelo todo de nuevo.
    Todos nos merecemos una segunda oportunidad. También tú.
    """
    if df_to_submit.shape == sample.shape:
        if df_to_submit.columns.all() == sample.columns.all():
            if df_to_submit.laptop_ID.all() == sample.laptop_ID.all():
                print("You're ready to submit!")
                df_to_submit.to_csv("submission.csv", index = False) #muy importante el index = False
                urllib.request.urlretrieve("https://www.mihaileric.com/static/evaluation-meme-e0a350f278a36346e6d46b139b1d0da0-ed51e.jpg", "gfg.png")
                img = Image.open("gfg.png")
                img.show()
            else:
                print("Check the ids and try again")
        else:
            print("Check the names of the columns and try again")
    else:
        print("Check the number of rows and/or columns and try again")
        print("\nMensaje secreto del TA: No me puedo creer que después de todo este notebook hayas hecho algún cambio en las filas de `test.csv`. Lloro.")

In [ ]:
chequeador(submission)